# 🎙️ Keerthana Gummuluri Venkata — Audio Analyst
## Multimodal Crime / Incident Report Analyzer

**Data Type:** Emergency audio calls / witness voice statements  
**Objective:** Convert audio → text → structured CSV with event, location, sentiment, urgency

### Pipeline:
1. Load audio files
2. Transcribe with OpenAI Whisper
3. Extract keywords & entities with spaCy
4. Sentiment/urgency analysis with HuggingFace
5. Output structured CSV


In [1]:
# =============================================
# CELL 1: Install Dependencies
# =============================================
!pip install -q openai-whisper spacy transformers torch torchaudio pandas
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 9.9 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 9.2 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 53.5 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# =============================================
# CELL 2: Import Libraries
# =============================================
import whisper
import spacy
import pandas as pd
import torch
import os
import warnings
warnings.filterwarnings('ignore')

from transformers import pipeline

# Load models
print("Loading Whisper model...")
whisper_model = whisper.load_model("base")  # Use 'small' or 'medium' for better accuracy

print("Loading spaCy NER model...")
nlp = spacy.load("en_core_web_sm")

print("Loading sentiment analysis pipeline...")
sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

print("All models loaded!")


Loading Whisper model...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 149MiB/s]


Loading spaCy NER model...
Loading sentiment analysis pipeline...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

All models loaded!


### Dataset: 911 Recordings — The First 6 Seconds

In [3]:
# =============================================
# CELL 3: Load 911 Audio Dataset
# =============================================
import os, glob

audio_dir = "/kaggle/input/datasets/louisteitelbaum/911-recordings-first-6-seconds"
all_audio_files = sorted(glob.glob(os.path.join(audio_dir, "**", "*.wav"), recursive=True))

print(f"Total audio files: {len(all_audio_files)}")
for f in all_audio_files[:5]:
    print(f"  {os.path.basename(f)} — {os.path.getsize(f):,} bytes")
if len(all_audio_files) > 5:
    print(f"  ... and {len(all_audio_files) - 5} more")


Total audio files: 707
  call_100_0.wav — 1,058,444 bytes
  call_101_0.wav — 1,058,444 bytes
  call_102_0.wav — 529,244 bytes
  call_103_0.wav — 1,058,444 bytes
  call_104_0.wav — 1,152,044 bytes
  ... and 702 more


In [4]:
# =============================================
# CELL 4: Transcription with Whisper
# =============================================

def transcribe_audio(file_path):
    """Transcribe audio file using OpenAI Whisper."""
    try:
        result = whisper_model.transcribe(file_path)
        return result["text"].strip()
    except Exception as e:
        print(f"Error transcribing {file_path}: {e}")
        return ""

# Transcribe all audio files from the Kaggle dataset
# Limit to first N files for speed (increase for full run)

files_to_process = all_audio_files

print(f"Transcribing {len(files_to_process)} / {len(all_audio_files)} audio files with Whisper...")
print("(Increase MAX_FILES above to process more)\n")

transcriptions = []
for i, filepath in enumerate(files_to_process, 1):
    filename = os.path.basename(filepath)
    print(f"  [{i}/{len(files_to_process)}] {filename}...", end=" ")

    transcript = transcribe_audio(filepath)
    call_id = f"C{i:03d}"

    transcriptions.append({
        "Call_ID": call_id,
        "Filename": filename,
        "Transcript": transcript
    })
    preview = transcript[:60] + "..." if len(transcript) > 60 else transcript
    print(f"Done: {preview}" if transcript else "No speech detected")

df_transcripts = pd.DataFrame(transcriptions)

# Remove rows with empty transcripts
empty_count = (df_transcripts["Transcript"] == "").sum()
if empty_count > 0:
    print(f"\nRemoving {empty_count} files with no detected speech.")
    df_transcripts = df_transcripts[df_transcripts["Transcript"] != ""].reset_index(drop=True)

print(f"\nSuccessfully transcribed: {len(df_transcripts)} audio files.")
df_transcripts.head(10)


Transcribing 707 / 707 audio files with Whisper...
(Increase MAX_FILES above to process more)

  [1/707] call_100_0.wav... Done: one different
  [2/707] call_101_0.wav... Done: This is the Blocopia Bank in 1302 with the Transnational Spe...
  [3/707] call_102_0.wav... Done: Yeah? Yeah. Yeah. Yeah. Yeah. Yeah.
  [4/707] call_103_0.wav... Done: Yes, we're at the Nelson building and there's some men here ...
  [5/707] call_104_0.wav... Done: Listen, we're playing really bad at this. Okay, so get on ou...
  [6/707] call_104_1.wav... Done: ...
  [7/707] call_105_0.wav... Done: I'm locking in the field and I'm looking for a champ. You're...
  [8/707] call_106_0.wav... Done: Marg massacre deBen
  [9/707] call_106_1.wav... Done: They'll remember to come out, and the others figured they're...
  [10/707] call_106_2.wav... Done: Okay, it's out for the study. Down there in the hall. We'll ...
  [11/707] call_106_3.wav... Done: Yes, that's two sub-pro here that's the 8 on the ground. Oka...
  [12/7

,Call_ID,Filename,Transcript
0,C001,call_100_0.wav,one different
1,C002,call_101_0.wav,This is the Blocopia Bank in 1302 with the Tra...
2,C003,call_102_0.wav,Yeah? Yeah. Yeah. Yeah. Yeah. Yeah.
3,C004,call_103_0.wav,"Yes, we're at the Nelson building and there's ..."
4,C005,call_104_0.wav,"Listen, we're playing really bad at this. Okay..."
5,C006,call_104_1.wav,...
6,C007,call_105_0.wav,I'm locking in the field and I'm looking for a...
7,C008,call_106_0.wav,Marg massacre deBen
8,C009,call_106_1.wav,"They'll remember to come out, and the others f..."
9,C010,call_106_2.wav,"Okay, it's out for the study. Down there in th..."


In [6]:
# =============================================
# CELL 5: Entity & Keyword Extraction with spaCy
# =============================================

# Define incident-related keyword categories
INCIDENT_KEYWORDS = {
    "fire": ["fire", "flame", "smoke", "burning", "blaze", "arson", "gas leak", "explosion", "exploded"],
    "accident": ["accident", "crash", "collision", "hit", "injured", "vehicle", "car", "wreck", "derail", "plane", "helicopter"],
    "theft": ["theft", "robbery", "robbed", "robbing", "stole", "stolen", "broke in", "burglar", "break-in", "breaking in", "kidnap"],
    "assault": ["fight", "assault", "attack", "knife", "weapon", "punch", "stabbing", "stabbed", "shot", "shooting", "gunshot", "gun", "killed", "murder", "homicide", "dead body", "strangled", "beat"],
    "medical": ["medical", "heart attack", "unconscious", "bleeding", "ambulance", "overdose", "not breathing", "choking", "drowning", "seizure", "CPR", "passed out", "pills"],
    "disturbance": ["disturbance", "noise", "drunk", "yelling", "suspicious", "trespassing", "harassing", "naked"]
}

def extract_incident_type(text):
    """Classify the incident type based on keywords."""
    text_lower = text.lower()
    scores = {}
    for incident_type, keywords in INCIDENT_KEYWORDS.items():
        score = sum(1 for kw in keywords if kw in text_lower)
        if score > 0:
            scores[incident_type] = score
    if scores:
        return max(scores, key=scores.get).title()
    return "Unknown"

def extract_entities(text):
    """Extract named entities using spaCy."""
    doc = nlp(text)
    locations = [ent.text for ent in doc.ents if ent.label_ in ["GPE", "LOC", "FAC"]]
    persons = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
    orgs = [ent.text for ent in doc.ents if ent.label_ == "ORG"]
    return {
        "locations": locations,
        "persons": persons,
        "organizations": orgs
    }

def extract_location_from_text(text):
    """Extract location mentions including street names."""
    entities = extract_entities(text)
    locations = entities["locations"]

    # Also look for common street patterns
    import re
    street_patterns = re.findall(
        r'(?:on|at|near|outside)\s+([A-Z][a-z]+(?:\s+(?:Street|St|Avenue|Ave|Road|Rd|Drive|Dr|Boulevard|Blvd|Lane|Ln))?(?:\s+(?:and|&)\s+[A-Z][a-z]+(?:\s+(?:Street|St|Avenue|Ave|Road|Rd))?)?)',
        text
    )
    all_locations = list(set(locations + street_patterns))
    return ", ".join(all_locations) if all_locations else "Unknown"

# Apply extraction
df_transcripts["Extracted_Event"] = df_transcripts["Transcript"].apply(extract_incident_type)
df_transcripts["Location"] = df_transcripts["Transcript"].apply(extract_location_from_text)

print("Extracted events and locations:")
df_transcripts[["Call_ID", "Extracted_Event", "Location"]]


Extracted events and locations:


,Call_ID,Extracted_Event,Location
0,C001,Unknown,Unknown
1,C002,Unknown,Unknown
2,C003,Unknown,Unknown
3,C004,Unknown,Unknown
4,C005,Unknown,Unknown
...,...,...,...
698,C703,Unknown,Unknown
699,C704,Unknown,Unknown
700,C705,Unknown,Unknown
701,C706,Assault,Unknown


In [7]:
# =============================================
# CELL 6: Sentiment & Urgency Analysis
# =============================================

URGENCY_KEYWORDS = {
    "high": ["immediately", "hurry", "emergency", "trapped", "dying", "bleeding", "help", "quickly", "now", "knife", "gun", "weapon", "shot", "shooting", "killed", "murder", "dead", "fire", "stabbed", "not breathing", "please", "robbery", "choking", "scared"],
    "medium": ["injured", "accident", "fight", "stolen", "broke", "damage", "crash", "suspicious", "drunk", "missing", "unconscious"],
    "low": ["noise", "smell", "minor", "someone", "report", "concern"]
}

def analyze_sentiment(text):
    """Analyze sentiment using HuggingFace transformer."""
    try:
        result = sentiment_analyzer(text[:512])[0]  # Truncate for model limit
        label = result["label"]
        score = result["score"]
        # Map: NEGATIVE with high confidence = Distressed
        if label == "NEGATIVE" and score > 0.8:
            return "Distressed"
        elif label == "NEGATIVE":
            return "Concerned"
        else:
            return "Calm"
    except:
        return "Unknown"
def analyze_sentiment_with_override(text):
    """Sentiment with keyword override for obvious emergencies."""
    base = analyze_sentiment(text)
    text_lower = text.lower()
    distress_words = ["shot", "killed", "murder", "dead", "fire", "stabbed", "trapped", "robbery", "shooting", "not breathing", "help me", "please hurry"]
    if any(w in text_lower for w in distress_words) and base == "Calm":
        return "Distressed"
    return base

def calculate_urgency(text):
    """Calculate urgency score (0-1) based on keywords and patterns."""
    text_lower = text.lower()
    score = 0
    max_score = 10

    # High urgency keywords
    for kw in URGENCY_KEYWORDS["high"]:
        if kw in text_lower:
            score += 3
    # Medium urgency keywords
    for kw in URGENCY_KEYWORDS["medium"]:
        if kw in text_lower:
            score += 2
    # Low urgency keywords
    for kw in URGENCY_KEYWORDS["low"]:
        if kw in text_lower:
            score += 1

    # Exclamation marks and capitalization
    score += text.count("!") * 0.5
    score += sum(1 for word in text.split() if word.isupper() and len(word) > 1) * 0.5

    return round(min(score / max_score, 1.0), 2)

# Apply sentiment and urgency analysis
df_transcripts["Sentiment"] = df_transcripts["Transcript"].apply(analyze_sentiment_with_override)
df_transcripts["Urgency_Score"] = df_transcripts["Transcript"].apply(calculate_urgency)

print("Sentiment & Urgency Analysis:")
df_transcripts[["Call_ID", "Sentiment", "Urgency_Score"]]


Sentiment & Urgency Analysis:


,Call_ID,Sentiment,Urgency_Score
0,C001,Calm,0.00
1,C002,Distressed,0.00
2,C003,Calm,0.00
3,C004,Calm,0.00
4,C005,Distressed,0.00
...,...,...,...
698,C703,Calm,0.65
699,C704,Distressed,0.00
700,C705,Calm,0.00
701,C706,Distressed,0.30


In [8]:
# =============================================
# CELL 7: Generate Final Structured CSV
# =============================================

# Select and order final columns
output_cols = ["Call_ID", "Transcript", "Extracted_Event", "Location", "Sentiment", "Urgency_Score"]
df_audio_output = df_transcripts[output_cols].copy()

# Save to CSV
os.makedirs("outputs", exist_ok=True)
df_audio_output.to_csv("outputs/audio_analyst_output.csv", index=False)

print("=" * 70)
print("AUDIO ANALYST - FINAL OUTPUT")
print("=" * 70)
print(f"Total calls processed: {len(df_audio_output)}")
print(f"Output saved to: outputs/audio_analyst_output.csv")
print("=" * 70)
df_audio_output


AUDIO ANALYST - FINAL OUTPUT
Total calls processed: 703
Output saved to: outputs/audio_analyst_output.csv


,Call_ID,Transcript,Extracted_Event,Location,Sentiment,Urgency_Score
0,C001,one different,Unknown,Unknown,Calm,0.00
1,C002,This is the Blocopia Bank in 1302 with the Tra...,Unknown,Unknown,Distressed,0.00
2,C003,Yeah? Yeah. Yeah. Yeah. Yeah. Yeah.,Unknown,Unknown,Calm,0.00
3,C004,"Yes, we're at the Nelson building and there's ...",Unknown,Unknown,Calm,0.00
4,C005,"Listen, we're playing really bad at this. Okay...",Unknown,Unknown,Distressed,0.00
...,...,...,...,...,...,...
698,C703,Now that is vac and sample the floor. So that ...,Unknown,Unknown,Calm,0.65
699,C704,"Oh, my mind's at the doctor. You're not at the...",Unknown,Unknown,Distressed,0.00
700,C705,I'm attracted to companies. I'm no different c...,Unknown,Unknown,Calm,0.00
701,C706,"Oh, there's a student. Who's been shot, man?",Assault,Unknown,Distressed,0.30


### ✅ Audio Analyst Complete!
**Output:** `outputs/audio_analyst_output.csv`

| Column | Description |
|--------|-------------|
| Call_ID | Unique identifier for each call |
| Transcript | Full transcribed text |
| Extracted_Event | Classified incident type |
| Location | Extracted location mentions |
| Sentiment | Calm / Concerned / Distressed |
| Urgency_Score | 0.0 - 1.0 urgency rating |

